# U-Net 256 aug_bc Augmentation Baseline

This notebook trains a U-Net baseline on VisA supervised defect segmentation and saves metrics for comparison.

Outputs are saved to Google Drive under `/content/drive/MyDrive/VisA_segmentation/visa_results/colab_runs/{EXPERIMENT_NAME}`.


In [ ]:
# Runtime setup
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/dansojo/Industrial-Defect-Segmentation.git'
REPO_DIR = '/content/Industrial-Defect-Segmentation'
DRIVE_DATA_ZIP = '/content/drive/MyDrive/VisA_segmentation/VisA.zip'
DATA_PARENT = '/content/data'
RESULTS_ROOT = '/content/drive/MyDrive/VisA_segmentation/visa_results/colab_runs'


In [ ]:
# Clone or update repository
import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())


In [ ]:
# Install dependencies
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'opencv-python-headless', 'pandas', 'matplotlib', 'tqdm'], check=True)


In [ ]:
# Prepare dataset from Drive zip into Colab local disk
import zipfile
from pathlib import Path

DATA_PARENT_PATH = Path(DATA_PARENT)
DATA_PARENT_PATH.mkdir(parents=True, exist_ok=True)
zip_path = Path(DRIVE_DATA_ZIP)


def find_data_root():
    candidates = [
        DATA_PARENT_PATH / 'VisA',
        DATA_PARENT_PATH / 'VisA_data',
        DATA_PARENT_PATH,
    ]
    return next(
        (p for p in candidates if (p / 'split_csv').exists() and (p / 'candle').exists()),
        None,
    )


DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    if not zip_path.exists():
        raise FileNotFoundError(f'Data zip not found: {zip_path}')
    print(f'Extracting {zip_path} to {DATA_PARENT_PATH} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_PARENT_PATH)
    DATA_ROOT = find_data_root()

if DATA_ROOT is None:
    children = [str(p) for p in DATA_PARENT_PATH.iterdir()]
    raise FileNotFoundError(f'Could not find expected VisA dataset root. children={children}')

print('DATA_ROOT:', DATA_ROOT)
print('Top-level sample:', sorted([p.name for p in DATA_ROOT.iterdir()])[:8])


In [ ]:
# Notebook-specific experiment name
EXPERIMENT_NAME = 'unet_256_aug_bc'
AUGMENTATION_NAME = 'aug_bc'


In [ ]:
# Imports
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.datasets.visa_dataset import VisASegmentationDataset
from src.losses.losses import build_loss
from src.metrics import binary_stats, merge_stats, metrics_from_stats
from src.models.unet import build_unet
from src.visualize import save_prediction_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Experiment config
AUGMENTATION_PARAMS = {
    'brightness_limit': 0.12,
    'contrast_limit': 0.12,
    'brightness_contrast_p': 0.5,
    'shift_limit': 0.03,
    'scale_limit': 0.08,
    'rotate_limit': 10,
    'geometric_p': 0.5,
}

INPUT_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-3
LOSS_NAME = 'bce_dice'
THRESHOLD = 0.5
NUM_WORKERS = 2

SPLIT_CSV = Path(REPO_DIR) / 'data' / 'splits' / 'visa_2cls_highshot_train_val_test.csv'
RUN_DIR = Path(RESULTS_ROOT) / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

config = {
    'experiment_name': EXPERIMENT_NAME,
    'augmentation': AUGMENTATION_NAME,
    'augmentation_params': AUGMENTATION_PARAMS,
    'input_size': INPUT_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'loss': LOSS_NAME,
    'threshold': THRESHOLD,
    'split_csv': str(SPLIT_CSV),
    'data_root': str(DATA_ROOT),
}
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print(json.dumps(config, indent=2))


In [ ]:
# Transforms
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def get_transforms(augmentation_name: str, split: str):
    transforms = [A.Resize(INPUT_SIZE, INPUT_SIZE)]

    if split == 'train' and augmentation_name in {'aug_bc', 'aug_mild'}:
        transforms.append(
            A.RandomBrightnessContrast(
                brightness_limit=AUGMENTATION_PARAMS['brightness_limit'],
                contrast_limit=AUGMENTATION_PARAMS['contrast_limit'],
                p=AUGMENTATION_PARAMS['brightness_contrast_p'],
            )
        )

    if split == 'train' and augmentation_name in {'aug_geo', 'aug_mild'}:
        transforms.append(
            A.ShiftScaleRotate(
                shift_limit=AUGMENTATION_PARAMS['shift_limit'],
                scale_limit=AUGMENTATION_PARAMS['scale_limit'],
                rotate_limit=AUGMENTATION_PARAMS['rotate_limit'],
                border_mode=0,
                p=AUGMENTATION_PARAMS['geometric_p'],
            )
        )

    transforms.extend([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])
    return A.Compose(transforms)


In [ ]:
# Datasets and dataloaders
train_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='train', transform=get_transforms(AUGMENTATION_NAME, 'train'))
val_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='val', transform=get_transforms(AUGMENTATION_NAME, 'val'))
test_ds = VisASegmentationDataset(DATA_ROOT, SPLIT_CSV, split='test', transform=get_transforms(AUGMENTATION_NAME, 'test'))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Train:', len(train_ds), 'Val:', len(val_ds), 'Test:', len(test_ds))
print(pd.read_csv(SPLIT_CSV).groupby(['split','label']).size().unstack(fill_value=0))


In [ ]:
# Training and evaluation helpers
model = build_unet().to(device)
criterion = build_loss(LOSS_NAME)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)


def train_one_epoch(loader):
    model.train()
    losses = []
    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(loader, save_grid_path=None):
    model.eval()
    losses = []
    stats_items = []
    category_stats = defaultdict(list)
    first_batch_saved = False
    inference_times = []

    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        logits = model(images)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        inference_times.append((time.perf_counter() - start) / images.shape[0])

        loss = criterion(logits, masks)
        losses.append(loss.item())
        stats_items.append(binary_stats(logits.cpu(), masks.cpu(), threshold=THRESHOLD))

        for idx, object_name in enumerate(batch['object']):
            item_stats = binary_stats(logits[idx:idx+1].cpu(), masks[idx:idx+1].cpu(), threshold=THRESHOLD)
            category_stats[object_name].append(item_stats)

        if save_grid_path is not None and not first_batch_saved:
            cpu_batch = {'image': batch['image'].cpu(), 'mask': batch['mask'].cpu()}
            save_prediction_grid(cpu_batch, logits.cpu(), save_grid_path, max_items=6)
            first_batch_saved = True

    overall_stats = merge_stats(stats_items)
    metrics = metrics_from_stats(overall_stats)
    metrics['loss'] = float(np.mean(losses))
    metrics['inference_time_ms'] = float(np.mean(inference_times) * 1000)

    category_rows = []
    for object_name, items in sorted(category_stats.items()):
        merged = merge_stats(items)
        row = {'object': object_name, **metrics_from_stats(merged)}
        category_rows.append(row)
    category_df = pd.DataFrame(category_rows)
    return metrics, category_df


In [ ]:
# Run training
history = []
best_dice = -1.0
best_path = RUN_DIR / 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(train_loader)
    val_metrics, val_category_df = evaluate(val_loader)
    scheduler.step(val_metrics['dice'])

    row = {'epoch': epoch, 'train_loss': train_loss, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / 'metrics.csv', index=False)
    val_category_df.to_csv(RUN_DIR / 'val_category_metrics_latest.csv', index=False)

    if val_metrics['dice'] > best_dice:
        best_dice = val_metrics['dice']
        torch.save({'model_state_dict': model.state_dict(), 'config': config, 'epoch': epoch, 'val_metrics': val_metrics}, best_path)
        val_category_df.to_csv(RUN_DIR / 'val_category_metrics_best.csv', index=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
        f"dice={val_metrics['dice']:.4f} | iou={val_metrics['iou']:.4f} | "
        f"precision={val_metrics['precision']:.4f} | recall={val_metrics['recall']:.4f} | "
        f"time={val_metrics['inference_time_ms']:.2f}ms/img"
    )

print('Best val dice:', best_dice)
print('Saved best checkpoint:', best_path)


In [ ]:
# Evaluate best checkpoint on validation and test, save comparable outputs
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

val_grid_path = RUN_DIR / 'val_prediction_grid.jpg'
test_grid_path = RUN_DIR / 'test_prediction_grid.jpg'
val_metrics, val_category_df = evaluate(val_loader, save_grid_path=val_grid_path)
test_metrics, test_category_df = evaluate(test_loader, save_grid_path=test_grid_path)

pd.DataFrame([{'split': 'val', **val_metrics}, {'split': 'test', **test_metrics}]).to_csv(RUN_DIR / 'final_metrics.csv', index=False)
val_category_df.to_csv(RUN_DIR / 'val_category_metrics.csv', index=False)
test_category_df.to_csv(RUN_DIR / 'test_category_metrics.csv', index=False)

print('Final overall metrics')
display(pd.DataFrame([{'split': 'val', **val_metrics}, {'split': 'test', **test_metrics}]))
print('Validation category metrics')
display(val_category_df.sort_values('dice'))
print('Test category metrics')
display(test_category_df.sort_values('dice'))
print('Saved outputs to:', RUN_DIR)
print('Validation grid:', val_grid_path)
print('Test grid:', test_grid_path)


In [ ]:
# Plot learning curves
history_df = pd.read_csv(RUN_DIR / 'metrics.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(history_df['epoch'], history_df['val_dice'], label='val_dice')
axes[1].plot(history_df['epoch'], history_df['val_iou'], label='val_iou')
axes[1].legend(); axes[1].set_title('Overlap')
axes[2].plot(history_df['epoch'], history_df['val_precision'], label='precision')
axes[2].plot(history_df['epoch'], history_df['val_recall'], label='recall')
axes[2].legend(); axes[2].set_title('Precision / Recall')
plt.tight_layout()
curve_path = RUN_DIR / 'learning_curves.png'
plt.savefig(curve_path, dpi=150)
plt.show()
print('Saved:', curve_path)


In [ ]:
# Display prediction grids
from IPython.display import Image, display

print('Validation prediction grid: original | GT green | prediction red')
display(Image(filename=str(RUN_DIR / 'val_prediction_grid.jpg')))
print('Test prediction grid: original | GT green | prediction red')
display(Image(filename=str(RUN_DIR / 'test_prediction_grid.jpg')))


## What to share back

After running this notebook, share these outputs:

- `final_metrics.csv`
- `val_category_metrics.csv`
- `test_category_metrics.csv`
- `metrics.csv`
- `learning_curves.png`
- `val_prediction_grid.jpg`
- `test_prediction_grid.jpg`

The most important comparison between notebooks is:

- validation Dice / IoU
- validation Precision / Recall
- category-wise Recall for small-defect categories (`macaroni1`, `macaroni2`)
- category-wise Precision for PCB categories
